# Alpha Distance — Google Colab L4

This notebook trains the largest practical model for a 24 GB NVIDIA L4 using local `/content` storage. Google Drive is mounted only for checkpoint snapshots approximately every 30 minutes and the final model.

The default configuration is a 1.06B-parameter Distance Alpha model with 2,048-token context. Adjust `DRIVE_ROOT` before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/alpha-distance-l4'
REPO = '/content/model-alpha'
LOCAL_OUT = '/content/alpha-distance-l4'
DRIVE_OUT = DRIVE_ROOT + '/checkpoints'
print('Set DRIVE_ROOT if needed:', DRIVE_ROOT)

In [ ]:
!rm -rf /content/model-alpha
!git clone --depth 1 https://github.com/thibodeaumaxim2-cyber/model-alpha.git /content/model-alpha
!python -m pip install -q -r /content/model-alpha/requirements.txt
!mkdir -p /content/alpha-distance-l4 /content/drive/MyDrive/alpha-distance-l4/checkpoints
!nvidia-smi

In [ ]:
# Hardware and memory check. The notebook stops here if Colab did not assign an L4.
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable.'
name = torch.cuda.get_device_name(0)
print(torch.__version__, name)
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
assert 'L4' in name, f'Expected an L4, got {name}'
print('BF16:', torch.cuda.is_bf16_supported())

## Data

Put a corpus file in Drive only once, or point `CORPUS` at an existing local file. The training script tokenizes into disk-backed chunks and does not load the full corpus into RAM. For best Drive performance, copy the corpus to `/content` before training.

In [ ]:
# Change this to your corpus. This example copies it once from Drive to local ephemeral storage.
CORPUS_DRIVE = DRIVE_ROOT + '/corpus.txt'
CORPUS = '/content/corpus.txt'
import os, shutil
if os.path.exists(CORPUS_DRIVE) and not os.path.exists(CORPUS):
    shutil.copyfile(CORPUS_DRIVE, CORPUS_DRIVE.replace(CORPUS_DRIVE, CORPUS))
print('Corpus:', CORPUS, os.path.exists(CORPUS))

In [ ]:
# One-step smoke test: small model only, validating CUDA, forward/backward, and checkpoint writing.
import subprocess, os
smoke = [
 'python', REPO + '/train_distance_alpha.py', '--corpus', CORPUS,
 '--output', '/content/alpha-smoke', '--steps', '1', '--save-every', '1',
 '--block-size', '512', '--dim', '256', '--heads', '8', '--layers', '4',
 '--clusters', '64', '--slots', '64', '--vocab-size', '32000',
 '--batch-size', '1', '--chunk-tokens', '100000', '--cache-chunks', '1']
if not os.path.exists(CORPUS): print('Place the corpus at', CORPUS, 'then rerun this cell.')
else:
    subprocess.run(smoke, check=True)
    assert os.path.getsize('/content/alpha-smoke/latest.pt') > 0
    print('Smoke test passed:', os.path.getsize('/content/alpha-smoke/latest.pt'), 'bytes')

## Main training

This uses the 1.06B configuration. The active checkpoint stays on local SSD. A background thread copies `latest.pt` to Drive at most once every 1,800 seconds; it never writes every step. The Drive copy is a recovery snapshot, not the active training path.

In [ ]:
import os, time, shutil, threading, subprocess, signal
os.makedirs(DRIVE_OUT, exist_ok=True)
stop_sync = False
def drive_sync():
    last = 0
    while not stop_sync:
        time.sleep(30)
        src = LOCAL_OUT + '/latest.pt'
        now = time.time()
        if os.path.exists(src) and now - last >= 1800:
            tmp = DRIVE_OUT + '/latest.pt.tmp'
            shutil.copyfile(src, tmp)
            if os.path.getsize(tmp) > 0:
                os.replace(tmp, DRIVE_OUT + '/latest.pt')
                last = now
                print('Drive snapshot:', time.ctime(now), os.path.getsize(src), 'bytes', flush=True)
threading.Thread(target=drive_sync, daemon=True).start()

cmd = [
 'python', REPO + '/train_distance_alpha.py', '--corpus', CORPUS,
 '--output', LOCAL_OUT, '--steps', '150000', '--save-every', '250',
 '--block-size', '2048', '--dim', '2048', '--heads', '16', '--layers', '18',
 '--clusters', '64', '--slots', '64', '--vocab-size', '32000',
 '--batch-size', '1', '--learning-rate', '1e-4', '--warmup-steps', '2000',
 '--schedule-steps', '150000', '--chunk-tokens', '500000', '--cache-chunks', '1']
print('Starting:', ' '.join(cmd))
try:
    subprocess.run(cmd, check=True)
finally:
    stop_sync = True
    src = LOCAL_OUT + '/latest.pt'
    if os.path.exists(src) and os.path.getsize(src) > 0:
        shutil.copyfile(src, DRIVE_OUT + '/latest.pt.final')
        shutil.copyfile(LOCAL_OUT + '/tokenizer.json', DRIVE_OUT + '/tokenizer.json')
        print('Final model copied to Drive.')

In [ ]:
# Resume after a Colab disconnect using the local checkpoint restored from Drive.
import os, shutil
os.makedirs(LOCAL_OUT, exist_ok=True)
src = DRIVE_OUT + '/latest.pt'
if os.path.exists(src):
    shutil.copyfile(src, LOCAL_OUT + '/latest.pt')
    if os.path.exists(DRIVE_OUT + '/tokenizer.json'):
        shutil.copyfile(DRIVE_OUT + '/tokenizer.json', LOCAL_OUT + '/tokenizer.json')
    print('Restored snapshot. Rerun the main cell with --resume added and keep the same architecture.')
else: print('No Drive snapshot found.')

## Model stats

- Parameters: **1,062,825,986**
- Vocabulary: **32,000**
- Context length: **2,048 tokens**
- Dimension: **2,048**
- Transformer layers: **18**
- Attention heads: **16**
- Distance clusters: **64**
- Memory slots per cluster: **64**
- Effective memory slots: **4,096**
- Precision: BF16 on L4
- Microbatch: 1, with disk-backed token sampling
- Checkpoint policy: local checkpoint every 250 steps; Drive snapshot at most every 30 minutes plus final copy

If the 1B smoke test runs out of memory, reduce `--cache-chunks` to `1` (already the default), then reduce `--block-size` only if the model code and checkpoint architecture are intentionally changed. Do not resume with different architecture settings.